# 06-gauss-svi — Gaussian theo lớp, học bằng SVI ở S0

**Câu hỏi**: bước 06 thực sự học gì cho mỗi lớp cũ, bằng thuật toán nào, đi được bao xa từ điểm khởi tạo, và vì sao det Σ học được nhỏ hơn 1? (D11, D23, D14, Q06.05)

### Nguồn gốc (provenance)

In commit và trạng thái dirty của code, rồi đặt `XLA_FLAGS` trước mọi lần import jax (mẫu ở `lab/README.vi.md` §11).

In [1]:
# cell 0: provenance (run first)
import os, subprocess, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "main.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path[:0] = [str(ROOT), str(ROOT / "lab")]
import replay; os.environ["XLA_FLAGS"] = replay._xla_flags()   # GPU deterministic mode before any jax import
git = lambda *a: subprocess.run(["git", *a], capture_output=True, text=True).stdout.strip()
print("commit", git("rev-parse", "--short", "HEAD"),
      "| code dirty", bool(git("status", "--porcelain", "--", ".", ":(exclude)runs", ":(exclude)notes",
                               ":(exclude)lab/*.ipynb", ":(exclude)lab/*.md")),
      "| XLA_FLAGS", os.environ["XLA_FLAGS"])

commit 831587e | code dirty False | XLA_FLAGS --xla_gpu_deterministic_ops=true


- **Kết quả**: commit `831587e`, code dirty False, XLA_FLAGS `--xla_gpu_deterministic_ops=true`.

## 1. Kiểm

### 1.1 Chạy lại bước 06 và xem từng khe đã đi bao xa

Dùng `replay.rerun`: nạp ảnh chụp sau bước 05 của `gc10_v2_ft_r0_s4`, chạy lại `s06_gauss_svi` (1000 bước Adam trên GPU, chế độ tất định), rồi so `params0` với `snap/06.pkl`. Quy tắc đạt: số thực lệch ≤ 1e-6 × max|giá trị| (QD21/C1). Đây là lần đầu `rerun` kiểm một bước chạy trên GPU; bước 04 và 05 chạy trên CPU.

Sau đó, với mỗi khe (slot) c = 0…9 của bộ phân lớp, in khoảng cách tới giá trị khởi tạo μ = 0, Σ = I: max|μ_c|, ‖μ_c‖, max|Σ_c − I|, và Σ_c có đối xứng không. Ở S0 chỉ có nhãn 0–4, nên khe 5–9 không nhận gradient. Dự đoán: khe 5–9 giữ nguyên μ = 0, Σ = I; khe 0–4 rời khởi tạo rất ít (D23 đo trên `fz`: max|μ| khoảng 0.03).

In [2]:
# cell 1: step 06 again (SVI on the GPU) from the snapshot of step 05, compared with snapshot 06 (replay.rerun, QD23);
#         then params0 per slot: distance from the initial values mu = 0, Sigma = I, and symmetry of Sigma
import numpy as np
RUN = "gc10_v2_ft_r0_s4"
out06, ok06 = replay.rerun(RUN, "06")
assert ok06, "step 06 does not reproduce the snapshot"
p0_11 = out06["params0"]
mu_11 = np.asarray(p0_11["class_means"], dtype=np.float64)          # (10, 64)
S_11 = np.asarray(p0_11["class_covs"], dtype=np.float64)            # (10, 64, 64)
I_11 = np.eye(S_11.shape[1])
print("params0:", {key: v.shape for key, v in p0_11.items()}, "| fit component:", out06["fit"])
for c in range(S_11.shape[0]):
    print(f"slot {c}: max|mu| {np.abs(mu_11[c]).max():.4f} | ||mu|| {np.linalg.norm(mu_11[c]):.4f} | "
          f"max|Sigma - I| {np.abs(S_11[c] - I_11).max():.4f} | symmetric {np.allclose(S_11[c], S_11[c].T)}")

/home/rito/miniforge3/envs/vbcgcd/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


rerun step 06 of gc10_v2_ft_r0_s4 | device gpu | GPU now NVIDIA GeForce RTX 4060 Laptop GPU, 621.01 | GPU of the run NVIDIA GeForce RTX 4060 Laptop GPU, 617.42 | driver differs: float outputs may move in the last bits
Initializing model


  0%|          | 4/1000 [00:02<09:28,  1.75it/s]

Step 0: loss = 125299.7188, train_acc = 625/764, 81.81%,  test_acc = 186/241, 77.18%, last_cov = 1.0, cov = 1.0


 10%|█         | 103/1000 [00:07<00:45, 19.71it/s]

Step 100: loss = 123232.2188, train_acc = 727/764, 95.16%,  test_acc = 207/241, 85.89%, last_cov = 1.0, cov = 1.0


... (14 outputs left out)

 90%|█████████ | 904/1000 [00:45<00:05, 19.01it/s]

Step 900: loss = 116957.0312, train_acc = 730/764, 95.55%,  test_acc = 206/241, 85.48%, last_cov = 1.0, cov = 1.0


100%|██████████| 1000/1000 [00:50<00:00, 19.86it/s]

  OK   params0      dict                   max|diff| 0
  OK   fit          str                    max|diff| nan
rerun step 06: OK (2/2 keys)
params0: {'class_covs': (10, 64, 64), 'class_means': (10, 64)} | fit component: fit_svi
slot 0: max|mu| 0.0324 | ||mu|| 0.0860 | max|Sigma - I| 0.0357 | symmetric True
slot 1: max|mu| 0.0323 | ||mu|| 0.0913 | max|Sigma - I| 0.0341 | symmetric True
slot 2: max|mu| 0.0329 | ||mu|| 0.0735 | max|Sigma - I| 0.0353 | symmetric True
slot 3: max|mu| 0.0324 | ||mu|| 0.0904 | max|Sigma - I| 0.0346 | symmetric True
slot 4: max|mu| 0.0323 | ||mu|| 0.1006 | max|Sigma - I| 0.0348 | symmetric True
slot 5: max|mu| 0.0000 | ||mu|| 0.0000 | max|Sigma - I| 0.0000 | symmetric True
slot 6: max|mu| 0.0000 | ||mu|| 0.0000 | max|Sigma - I| 0.0000 | symmetric True
slot 7: max|mu| 0.0000 | ||mu|| 0.0000 | max|Sigma - I| 0.0000 | symmetric True
slot 8: max|mu| 0.0000 | ||mu|| 0.0000 | max|Sigma - I| 0.0000 | symmetric True
slot 9: max|mu| 0.0000 | ||mu|| 0.0000 | max|Sigma 

- **Kết quả**:
  - `rerun` bước 06 đạt 2/2 khoá. `params0` trùng ảnh chụp tuyệt đối (max|diff| = 0), dù driver GPU đã đổi từ 617.42 sang 621.01. SVI mất 52 giây cho 1000 bước.
  - Log huấn luyện (độ chính xác trên S0 và trên test S0 theo bước):
    - bước 0: 625/764 (81.81%), test 186/241 (77.18%);
    - bước 100: 727/764 (95.16%), test 207/241 (85.89%);
    - bước 200–900: train 95.42–95.68%, test 204–206/241 (84.65–85.48%);
    - loss giảm từ 125300 xuống khoảng 116500–117000, tức khoảng 7%.
  - Khe 0–4 (lớp cũ): max|μ| 0.0323–0.0329, ‖μ‖ 0.074–0.101, max|Σ − I| 0.034–0.036; Σ đối xứng.
  - Khe 5–9 (lớp mới): μ = 0, Σ = I, không đổi.
- **Ý nghĩa**:
  - Bước 06 tất định từng bit trên GPU khi bật chế độ tất định (QD21).
  - Đúng như dự đoán: khe 5–9 giữ nguyên giá trị khởi tạo, nên ở đầu phiên S1 mọi lớp mới bắt đầu từ N(0, I). Khe 0–4 rời khởi tạo đúng cỡ D23 đo trên `fz`: max|μ| khoảng 0.03, xấp xỉ tổng learning rate sau 1000 bước (0.0334, §2.2).
  - Độ chính xác đạt gần mức cuối ngay từ đầu. Sau 1 bước đã có 81.8% trên S0. Sau 100 bước, mức test S0 đứng yên ở khoảng 85%, trong khi loss vẫn giảm tới bước 900. Nghĩa là 900 bước sau gần như không đổi quyết định của bộ phân lớp.
  - [SUY LUẬN, kiểm ở §3.1] Vì sao 1 bước đã đủ: với Σ_c ≈ I, log N(x | μ_c, I) = −½‖x − μ_c‖² + hằng số, nên argmax theo c bằng argmax của μ_cᵀx − ½‖μ_c‖². Khi μ_c rất nhỏ, ½‖μ_c‖² không đáng kể, nên lớp được chọn là lớp có μ_cᵀx lớn nhất. Adam ở bước đầu cập nhật mỗi phần tử một lượng gần bằng learning rate η, theo dấu của gradient. Gradient của log-likelihood theo μ_c tại μ_c = 0 tỉ lệ với tổng các x_i của lớp c trong minibatch, nên μ_c ≈ η · sign(x̄_c). Hệ số η không đổi argmax, nên bộ phân lớp sau 1 bước là "argmax_c sign(x̄_c)ᵀx", một bộ phân lớp tuyến tính chỉ dùng dấu của tâm lớp.

## 2. Code

### 2.1 SVI trong code: mô hình, guide, vòng lặp

In nguyên văn bằng `inspect.getsource`:
- `replay.s06_gauss_svi` và `replay.fit_svi`: bước 06 gọi `run_inference` trên `z_s0` với nhãn S0;
- `MNGMMClassifier.model`: μ_c và Σ_c là `numpyro.param`, khởi tạo 0 và I; mỗi bước chỉ lấy minibatch 128 ảnh (`plate(..., subsample_size=self.batch_size)`);
- `MNGMMClassifier.run_inference`: guide là `lambda: None`, loss `Trace_ELBO`, Adam với lịch learning rate, khởi tạo bằng `PRNGKey(0)`, luật dừng sớm, dòng log mỗi 100 bước.

**Cách làm** (D11): khi mô hình không có biến ngẫu nhiên tiềm ẩn nào (mọi tham số là `param`, không phải `sample`) và guide rỗng, ELBO = E_q[log p(x, z)] − E_q[log q(z)] chỉ còn log p(x | μ, Σ). Vậy Trace_ELBO là log-likelihood âm, và SVI chính là MLE bằng gradient ngẫu nhiên (stochastic gradient), không có prior, không có số hạng KL (Hoffman et al. 2013 cho dạng tổng quát của SVI).

In [3]:
# cell 2: the SVI code as it is (inspect.getsource): s06_gauss_svi, fit_svi, MNGMMClassifier.model, run_inference
import inspect
from classifier.mngmm import MNGMMClassifier
for f_22 in (replay.s06_gauss_svi, replay.fit_svi, MNGMMClassifier.model, MNGMMClassifier.run_inference):
    print(f"# ---- {Path(inspect.getsourcefile(f_22)).name}: {f_22.__qualname__}\n{inspect.getsource(f_22)}")

# ---- replay.py: s06_gauss_svi
def s06_gauss_svi(st):
    """06 Class Gaussians of S0 (mngmm.train at stage 0), default early-stop rule (inactive at S0).
    In: z_s0, s0_y; z_test0, true_test0_y (log lines only). Out: params0; fit (estimator for 11 and 13)."""
    clf, a = _clf(st), st["args"]
    stop_default(clf, a)
    clf.global_params, clf.label_offset, clf.n_seen = None, 0, clf.num_base
    p = fit_svi(clf, st["z_s0"], np.asarray(st["s0_y"]).astype(int), st["z_test0"], st["true_test0_y"],
                "stage_0_Flearning", False)
    return dict(params0=_np(p), fit=fit_svi.__name__)

# ---- replay.py: fit_svi
def fit_svi(clf, z, y, z_test, y_test, prefix, correct_sf):
    """Estimator: class Gaussians by SVI (mngmm.run_inference, as in train(); PRNGKey(0) inside, no global RNG).
    The test data only feed the log lines."""
    import jax.numpy as jnp
    return clf.run_inference(jnp.array(z), jnp.array(y), jnp.array(z_test), jnp.array(y_test),
                             lo

- **Kết quả**:
  - `s06_gauss_svi` đặt `global_params = None`, `label_offset = 0`, rồi gọi `fit_svi`, tức `run_inference`, trên `z_s0` (764 × 64) với nhãn S0. Tập test S0 chỉ dùng để in log.
  - `model`: khi `global_params` là None, `class_means` khởi tạo bằng 0 và `class_covs` bằng I cho cả 10 khe; cả hai là `numpyro.param`, không phải `numpyro.sample`. Mỗi bước, `plate(..., subsample_size=128)` bốc 128 chỉ số hàng và tính log N(x | μ_y, Σ_y) cho 128 ảnh đó. Khối `cov_prior` chỉ chạy khi `kappa > 0` (mặc định 0, CT1).
  - `run_inference`:
    - lịch learning rate gồm hai đoạn: tăng tuyến tính từ `init_lr` lên `10·init_lr` trong 100 bước, sau đó giảm theo hàm mũ, hệ số 0.85 mỗi 500 bước;
    - guide là `lambda *args, **kwargs: None`;
    - `SVI(..., loss=Trace_ELBO())`, khởi tạo bằng `random.PRNGKey(0)`;
    - mỗi bước, luật dừng sớm được kiểm trước khi cập nhật, nhưng chỉ có thể kích hoạt khi bước > 1000/3;
    - in log mỗi 100 bước; gặp NaN thì quay về trạng thái trước.
- **Ý nghĩa**:
  - Xác nhận D11 bằng code: không có biến tiềm ẩn và guide rỗng, nên Trace_ELBO chỉ còn log-likelihood âm. "SVI" ở đây là MLE bằng Adam trên minibatch 128 ảnh, không có prior N(0, I) và không có số hạng KL như Eq. 7 của paper mô tả. N(0, I) chỉ còn là giá trị khởi tạo.
  - Loss in trong log (khoảng 1.2 × 10⁵) là log-likelihood âm của minibatch, nhân lên theo tỉ lệ 764/128 để ước lượng cho cả tập. Vì vậy loss dao động lên xuống giữa các dòng log, ví dụ bước 600 là 117943, bước 700 là 118062.
  - `PRNGKey(0)` cố định dãy chỉ số minibatch cho mọi seed. Nhưng chỉ số ở đây là chỉ số hàng, và bước 04 xáo hàng theo seed. [SUY LUẬN] Vậy cùng một dãy chỉ số sẽ bốc ra những ảnh khác nhau ở mỗi seed. Đây là nguồn khác biệt thứ hai của `params0` giữa các seed, bên cạnh cơ sở PCA (05 §5.1). §4.1 đo tổng hiệu ứng của hai nguồn này.
  - Ở S0, `calculate_metrics_on_covariances` trả về False ngay khi `global_params` là None, nên luật dừng không chạy. Đây là lý do log in `last_cov = 1.0, cov = 1.0` ở mọi dòng.

### 2.2 Lịch learning rate và tổng ngân sách bước

**Cách làm**: Adam cập nhật mỗi tham số một lượng xấp xỉ bằng learning rate α_t, bị chặn trên cỡ α_t (Kingma & Ba 2015, §2.1). Vì vậy tham số đi xa nhất được sau T bước khoảng Σ_t α_t, khi gradient của nó giữ nguyên dấu suốt T bước. Tính lại lịch bằng chính `optax`, với `init_lr` của run (`args.lr`), rồi so tổng đó với max|μ_c| và max|Σ_c − I| ở §1.1.

Ví dụ tính tay cho đoạn tăng tuyến tính, với α_0 = 4·10⁻⁶ và α tăng 3.6·10⁻⁷ mỗi bước:
1. α_t = 4·10⁻⁶ + 3.6·10⁻⁵ · t/100, với t = 0…99;
2. Σ α_t = 100 × 4·10⁻⁶ + 3.6·10⁻⁵ × (0 + 1 + … + 99)/100 = 4·10⁻⁴ + 3.6·10⁻⁵ × 49.5 = 4·10⁻⁴ + 1.782·10⁻³ = 2.182·10⁻³.

Dự đoán: Σ_{t<1000} α_t ≈ 0.0334 (D23), và max|μ| = 0.0329 ở §1.1 nằm sát con số này.

In [4]:
# cell 3: the learning-rate schedule of run_inference rebuilt with optax (same calls as the code, init_lr of the run)
#         and the total step budget, against how far params0 moved (cell 1)
import optax
lr_33 = replay.snapshots(RUN)["04"]["meta"]["args"]["lr"]          # init_lr of the run (4e-6)
sched_33 = optax.join_schedules(schedules=[optax.linear_schedule(init_value=lr_33, end_value=lr_33 * 10, transition_steps=100),
                                           optax.exponential_decay(init_value=lr_33 * 10, transition_steps=500, decay_rate=0.85)],
                                boundaries=[100])
v_33 = np.array([float(sched_33(t)) for t in range(1000)])
cum_33 = np.cumsum(v_33)
print("init_lr", lr_33, "| lr at step 0, 50, 99, 100, 600, 999:", [f"{v_33[t]:.3g}" for t in (0, 50, 99, 100, 600, 999)])
print("sum of lr over the first 100, 334, 500, 1000 steps:", [round(float(cum_33[t - 1]), 5) for t in (100, 334, 500, 1000)])
print("slots 0-4: max |mu|", round(float(np.abs(mu_11[:5]).max()), 4), "| max |Sigma - I|", round(float(np.abs(S_11[:5] - I_11).max()), 4))

init_lr 4e-06 | lr at step 0, 50, 99, 100, 600, 999: ['4e-06', '2.2e-05', '3.96e-05', '4e-05', '3.4e-05', '2.99e-05']
sum of lr over the first 100, 334, 500, 1000 steps: [0.00218, 0.0112, 0.01719, 0.0334]
slots 0-4: max |mu| 0.0329 | max |Sigma - I| 0.0357


- **Kết quả**:
  - `init_lr` = 4·10⁻⁶. Learning rate ở bước 0, 50, 99, 100, 600, 999 lần lượt là 4·10⁻⁶, 2.2·10⁻⁵, 3.96·10⁻⁵, 4·10⁻⁵, 3.4·10⁻⁵, 2.99·10⁻⁵.
  - Tổng learning rate sau 100, 334, 500, 1000 bước: 0.00218, 0.0112, 0.01719, 0.0334.
  - Khe 0–4: max|μ| = 0.0329, max|Σ − I| = 0.0357.
- **Ý nghĩa**:
  - Tổng sau 100 bước (0.00218) trùng với phép tính tay ở trên (2.182·10⁻³). Tổng sau 1000 bước là 0.0334, đúng như D23.
  - max|μ| = 0.0329 bằng 98.5% tổng ngân sách (0.0329/0.0334, tự chia): ít nhất một phần tử của μ đã đi theo một hướng gần như suốt 1000 bước, với tốc độ tối đa. Tham số bị giới hạn bởi ngân sách bước, không phải bởi dữ liệu.
  - max|Σ − I| = 0.0357 vượt tổng learning rate một chút (khoảng 7%). Điều này không mâu thuẫn: một bước của Adam có thể lớn hơn α_t khi độ lớn gradient giảm dần, vì khi đó m_t/√v_t > 1; chặn trên trong trường hợp xấu nhất là khoảng 3.16·α_t (Kingma & Ba 2015, §2.1).
  - Để so: tâm dữ liệu của mỗi lớp nằm cách gốc toạ độ cỡ vài đơn vị trở lên (§3.1 đo), nên quãng đường tối đa 0.033 mỗi phần tử là rất nhỏ so với khoảng cần đi.

## 3. params0 so với dữ liệu

### 3.1 μ và Σ học được so với tâm và hiệp phương sai mẫu; bộ phân lớp "dấu của tâm"

Với mỗi lớp cũ c, trên `z_s0` của ảnh chụp:
- tâm mẫu x̄_c, và tỉ số r_μ = ‖μ_c − x̄_c‖ / ‖x̄_c‖ (r_μ ≈ 1 nghĩa là μ_c gần như chưa rời gốc; D23);
- cos(μ_c, x̄_c) và cos(μ_c, sign(x̄_c)): μ_c chỉ theo hướng của tâm, hay theo hướng dấu của tâm;
- khoảng trị riêng của Σ_c so với của hiệp phương sai mẫu S_c (05 §3.2).

Sau đó kiểm giả thuyết ở ô Sau của cell 1: so bộ phân lớp của `params0` (argmax log N(x | μ_c, Σ_c) trên đủ 10 khe, như `_predict`) với bộ phân lớp "dấu của tâm" ŷ(x) = argmax_c sign(x̄_c)ᵀx trên 5 lớp cũ.

**Cách làm**: log-mật độ tính bằng numpy float64, qua phân rã Cholesky; pipeline dùng JAX float32 nên có thể lệch ở các ca gần hoà. Độ chính xác của `params0` trên test S0 phải ra 206/241, bằng S0_All của run (85.48%); đây là phép kiểm cho cách tính. Đại lượng so sánh: độ chính xác trên S0 và test S0, và tỉ lệ ảnh test mà hai bộ phân lớp cho cùng nhãn.

Dự đoán [SUY LUẬN]: r_μ gần 1; cos(μ_c, sign(x̄_c)) cao hơn cos(μ_c, x̄_c); hai bộ phân lớp trùng nhau trên phần lớn ảnh test. Độ chính xác của bộ "dấu của tâm" gần với số ở bước 0 (625/764, 186/241). Bước 0 dùng dấu của tâm minibatch 128 ảnh, không phải tâm toàn tập, nên hai số không cần trùng.

In [5]:
# cell 4: params0 against the data (z_s0 of the snapshot): learned mean vs sample centre and its sign, learned vs sample
#         covariance spectra; then the params0 classifier (argmax over all 10 slots, numpy float64) vs the "sign of the
#         centre" classifier argmax_c sign(xbar_c) . x on the 5 old classes
blobs_44 = replay.snapshots(RUN)
z_44, y_44 = np.asarray(blobs_44["05"]["out"]["z_s0"], np.float64), np.asarray(blobs_44["04"]["out"]["s0_y"]).astype(int)
zt_44, yt_44 = np.asarray(blobs_44["05"]["out"]["z_test0"], np.float64), np.asarray(blobs_44["04"]["out"]["true_test0_y"]).astype(int)
xbar_44 = np.stack([z_44[y_44 == c].mean(0) for c in range(5)])
cos_44 = lambda a, b: float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
for c in range(5):
    Sc = np.cov(z_44[y_44 == c].T)
    print(f"class {c}: ||xbar|| {np.linalg.norm(xbar_44[c]):.3f} | ||mu|| {np.linalg.norm(mu_11[c]):.4f} | "
          f"r_mu {np.linalg.norm(mu_11[c] - xbar_44[c]) / np.linalg.norm(xbar_44[c]):.4f} | cos(mu, xbar) {cos_44(mu_11[c], xbar_44[c]):.3f} | "
          f"cos(mu, sign xbar) {cos_44(mu_11[c], np.sign(xbar_44[c])):.3f} | eig Sigma [{np.linalg.eigvalsh(S_11[c]).min():.3f}, "
          f"{np.linalg.eigvalsh(S_11[c]).max():.3f}] | eig S [{np.linalg.eigvalsh(Sc).min():.4f}, {np.linalg.eigvalsh(Sc).max():.2f}]")
def logpdf_44(X, mu, S):                                            # log N(x | mu, S) for every row of X
    L = np.linalg.cholesky(S)
    sol = np.linalg.solve(L, (X - mu).T)
    return -0.5 * (sol ** 2).sum(0) - np.log(np.diag(L)).sum() - 0.5 * X.shape[1] * np.log(2 * np.pi)
pred_p0 = lambda X: np.stack([logpdf_44(X, mu_11[c], S_11[c]) for c in range(10)], 1).argmax(1)
pred_sg = lambda X: (X @ np.sign(xbar_44).T).argmax(1)
for name_44, X, yy in (("S0", z_44, y_44), ("test S0", zt_44, yt_44)):
    a, b = pred_p0(X), pred_sg(X)
    print(f"{name_44}: params0 {int((a == yy).sum())}/{len(yy)} | sign of centre {int((b == yy).sum())}/{len(yy)} | "
          f"same label {int((a == b).sum())}/{len(yy)} | params0 predicts slot >= 5: {int((a >= 5).sum())}")

class 0: ||xbar|| 10.762 | ||mu|| 0.0860 | r_mu 0.9943 | cos(mu, xbar) 0.720 | cos(mu, sign xbar) 0.610 | eig Sigma [0.824, 1.318] | eig S [0.0162, 27.85]
class 1: ||xbar|| 8.046 | ||mu|| 0.0913 | r_mu 0.9921 | cos(mu, xbar) 0.702 | cos(mu, sign xbar) 0.683 | eig Sigma [0.832, 1.267] | eig S [0.0343, 32.28]
class 2: ||xbar|| 10.731 | ||mu|| 0.0735 | r_mu 0.9955 | cos(mu, xbar) 0.658 | cos(mu, sign xbar) 0.592 | eig Sigma [0.831, 1.324] | eig S [0.0206, 17.47]
class 3: ||xbar|| 9.748 | ||mu|| 0.0904 | r_mu 0.9942 | cos(mu, xbar) 0.632 | cos(mu, sign xbar) 0.673 | eig Sigma [0.827, 1.271] | eig S [0.0227, 21.91]
class 4: ||xbar|| 9.342 | ||mu|| 0.1006 | r_mu 0.9919 | cos(mu, xbar) 0.754 | cos(mu, sign xbar) 0.678 | eig Sigma [0.844, 1.303] | eig S [0.0117, 31.56]
S0: params0 732/764 | sign of centre 673/764 | same label 684/764 | params0 predicts slot >= 5: 0
test S0: params0 206/241 | sign of centre 191/241 | same label 208/241 | params0 predicts slot >= 5: 0


- **Kết quả**:

  | lớp | ‖x̄‖ | ‖μ‖ | r_μ | cos(μ, x̄) | cos(μ, sign x̄) | trị riêng Σ | trị riêng S |
  |---|---|---|---|---|---|---|---|
  | 0 punching_hole | 10.76 | 0.086 | 0.994 | 0.720 | 0.610 | 0.824–1.318 | 0.016–27.9 |
  | 1 welding_line | 8.05 | 0.091 | 0.992 | 0.702 | 0.683 | 0.832–1.267 | 0.034–32.3 |
  | 2 crescent_gap | 10.73 | 0.074 | 0.996 | 0.658 | 0.592 | 0.831–1.324 | 0.021–17.5 |
  | 3 water_spot | 9.75 | 0.090 | 0.994 | 0.632 | 0.673 | 0.827–1.271 | 0.023–21.9 |
  | 4 oil_spot | 9.34 | 0.101 | 0.992 | 0.754 | 0.678 | 0.844–1.303 | 0.012–31.6 |

  Bộ phân lớp (số ảnh đúng):
  - S0: `params0` 732/764, "dấu của tâm" 673/764, cùng nhãn 684/764;
  - test S0: `params0` 206/241, "dấu của tâm" 191/241, cùng nhãn 208/241;
  - `params0` không đoán khe 5–9 cho ảnh nào.
- **Ý nghĩa**:
  - Phép kiểm đạt: `params0` tính bằng numpy cho 206/241 trên test S0, đúng S0_All của run.
  - D23 đúng trên `ft` r0: ‖μ_c‖ khoảng 0.07–0.10, trong khi tâm mẫu cách gốc 8–11 đơn vị, nên r_μ = 0.992–0.996. Σ_c có trị riêng 0.82–1.32, gần I, trong khi hiệp phương sai mẫu có trị riêng trải từ 0.01 tới 32. Gaussian học được không khớp với dữ liệu. Bộ phân lớp gồm điểm khởi tạo cộng một nhiễu loạn nhỏ, và nhiễu loạn đó mang hướng của lớp.
  - Dự đoán "μ theo hướng dấu của tâm" sai ở 4/5 lớp: cos(μ, x̄) = 0.63–0.75 cao hơn cos(μ, sign x̄) = 0.59–0.68, trừ water_spot. Bộ "dấu của tâm" cũng kém hơn `params0` 15 ảnh trên test S0 (191 so với 206), và hai bộ chỉ cùng nhãn ở 208/241 ảnh. Vậy `params0` không phải bộ phân lớp "dấu của tâm".
  - [SUY LUẬN] Lý do có thể: Adam chia gradient cho căn bậc hai trung bình bình phương của chính nó. Phần tử nào có tâm lớn so với nhiễu minibatch thì giữ dấu ổn định và đi đủ ngân sách; phần tử nào có tâm nhỏ thì đổi dấu thường xuyên và đi ít. Như vậy μ_c tỉ lệ với một kiểu tỉ số tín hiệu/nhiễu của từng chiều, nên cos(μ, x̄) cao hơn cos(μ, sign x̄). Chưa kiểm.
  - Sửa lại ý ở ô Sau của cell 1: 900 bước cuối không đổi độ chính xác trên test S0, nhưng 100 bước đầu thì có (từ 186 lên 207/241). Bộ phân lớp được định hình chủ yếu trong pha tăng learning rate.
  - Ở S0, khe 5–9 (N(0, I)) không thắng ảnh nào, khớp suy luận của D24 rằng khe chưa huấn luyện hiếm khi thắng khi dùng SVI.

### 3.2 Vì sao det Σ < 1 (Q06.05)

05 §3.2 cho thấy 24–32/64 hướng của mỗi lớp cũ có phương sai trong lớp lớn hơn 1, nên thang đo của dữ liệu không giải thích được việc det Σ học được < 1. Ứng viên: phần ngoài đường chéo của Σ.

**Cách làm**: đặt Σ_c = I + E_c. Khai triển log det (Taylor của log(1 + λ) theo các trị riêng λ của E):
log det(I + E) = tr E − ½ tr(E²) + ⅓ tr(E³) − …

Với E đối xứng, tr(E²) = ‖E‖²_F, tách được thành phần đường chéo Σ_j E_jj² và phần ngoài đường chéo Σ_{j≠k} E_jk². In cho từng lớp cũ: tr E, ½‖E‖²_F của phần đường chéo, ½‖E‖²_F của phần ngoài đường chéo, khai triển bậc 2 và bậc 3, log det đúng (`slogdet`), và det. Cuối cùng in det trung bình của 5 lớp, là đại lượng mà luật dừng dùng (`dets[0]`); notebook cũ ghi 0.3707.

Ví dụ với số nhỏ: E = [[0.03, 0.03], [0.03, −0.03]] có tr E = 0, ‖E‖²_F = 4 × 0.0009 = 0.0036, nên log det ≈ 0 − 0.0018 = −0.0018. Kiểm trực tiếp: det = 1.03 × 0.97 − 0.03² = 0.9991 − 0.0009 = 0.9982, và log 0.9982 ≈ −0.0018. Dù tr E = 0, phần ngoài đường chéo vẫn kéo log det xuống.

In [6]:
# cell 5: Q06.05, log det Sigma_c = log det(I + E) split into tr E, the diagonal and off-diagonal halves of ||E||_F^2,
#         and the 2nd- and 3rd-order expansions, against the exact slogdet; then the mean det used by the early-stop rule
rows_55 = []
for c in range(5):
    E = S_11[c] - I_11
    off = E - np.diag(np.diag(E))
    trE, half_diag, half_off = np.trace(E), 0.5 * (np.diag(E) ** 2).sum(), 0.5 * (off ** 2).sum()
    order2 = trE - half_diag - half_off
    order3 = order2 + np.trace(E @ E @ E) / 3
    rows_55.append({"tr E": trE, "1/2 ||diag E||^2": half_diag, "1/2 ||offdiag E||^2": half_off, "order 2": order2, "order 3": order3,
                    "log det (exact)": np.linalg.slogdet(S_11[c])[1], "det": np.linalg.det(S_11[c]),
                    "offdiag share of |E|_F^2": (off ** 2).sum() / (E ** 2).sum()})
import pandas as pd
print(pd.DataFrame(rows_55, index=[f"class {c}" for c in range(5)]).round(4).to_string())
print("mean det of the 5 old classes (dets[0] of the early-stop rule):", round(float(np.mean([r["det"] for r in rows_55])), 4))

           tr E  1/2 ||diag E||^2  1/2 ||offdiag E||^2  order 2  order 3  log det (exact)     det  offdiag share of |E|_F^2
class 0 -0.9346            0.0271               0.4715  -1.4332  -1.4075          -1.4166  0.2425                    0.9456
class 1 -0.4025            0.0234               0.4232  -0.8491  -0.8322          -0.8383  0.4325                    0.9476
class 2 -0.6277            0.0272               0.5096  -1.1645  -1.1231          -1.1346  0.3215                    0.9494
class 3 -0.3814            0.0258               0.4749  -0.8821  -0.8588          -0.8664  0.4204                    0.9486
class 4 -0.3455            0.0256               0.4851  -0.8561  -0.8197          -0.8291  0.4365                    0.9499
mean det of the 5 old classes (dets[0] of the early-stop rule): 0.3707


- **Kết quả**:

  | lớp | tr E | ½‖đường chéo E‖² | ½‖ngoài chéo E‖² | khai triển bậc 2 | bậc 3 | log det đúng | det | phần ngoài chéo trong ‖E‖²_F |
  |---|---|---|---|---|---|---|---|---|
  | 0 | −0.935 | 0.027 | 0.472 | −1.433 | −1.408 | −1.417 | 0.243 | 0.946 |
  | 1 | −0.403 | 0.023 | 0.423 | −0.849 | −0.832 | −0.838 | 0.433 | 0.948 |
  | 2 | −0.628 | 0.027 | 0.510 | −1.165 | −1.123 | −1.135 | 0.322 | 0.949 |
  | 3 | −0.381 | 0.026 | 0.475 | −0.882 | −0.859 | −0.866 | 0.420 | 0.949 |
  | 4 | −0.346 | 0.026 | 0.485 | −0.856 | −0.820 | −0.829 | 0.437 | 0.950 |

  det trung bình của 5 lớp cũ: 0.3707.
- **Ý nghĩa**:
  - det trung bình 0.3707 trùng với số notebook cũ đo trên cùng ảnh chụp.
  - Khai triển bậc 2 lệch log det đúng không quá 0.03, bậc 3 không quá 0.012, nên công thức log det ≈ tr E − ½‖E‖²_F dùng được để tách nguồn.
  - Trả lời Q06.05: hai phần cùng kéo log det xuống.
    - Phần ngoài đường chéo: ½‖E_ngoài chéo‖² = 0.42–0.51 ở cả 5 lớp, gần như bằng nhau. Nó chiếm 95% ‖E‖²_F. Với 4032 phần tử, độ lớn trung bình mỗi phần tử khoảng √(2 × 0.47 / 4032) ≈ 0.015 (tự tính), khoảng một nửa ngân sách bước. Phần này có mặt bất kể dữ liệu, vì Adam di chuyển mọi phần tử nhận gradient khác 0.
    - Vết: tr E = −0.35 tới −0.93, tức trung bình mỗi phần tử đường chéo giảm 0.005–0.015, trong khi mỗi phần tử đường chéo dịch cỡ ±0.03 (½‖đường chéo‖² ≈ 0.026). Vậy các phần tử đường chéo đi theo cả hai hướng, nghiêng nhẹ về phía giảm.
    - Ở lớp 1, 3, 4, phần ngoài đường chéo đóng góp nhiều hơn vết. Ở lớp 0 và 2, vết đóng góp nhiều hơn.
  - Giả thuyết đúng một phần: phần ngoài đường chéo là một nguồn lớn, có hệ thống, nhưng vết âm cũng đáng kể. [SUY LUẬN] Vì sao vết âm dù 24–32/64 hướng có phương sai trong lớp lớn hơn 1 (05 §3.2): có thể do số hướng có phương sai nhỏ hơn 1 nhiều hơn một chút (32–40), cộng với nhiễu minibatch. Chưa tách được ở đây.
  - Hệ quả cho luật dừng (D14): det của lớp cũ (0.37) và ngưỡng suy ra từ nó (F: 0.185, S: 0.524) phần lớn do ngân sách tối ưu quyết định, không do thang đo của dữ liệu. Điều này ủng hộ suy luận của D23 rằng luật dừng thực chất là luật đếm bước. Kiểm ở 13.

## 4. Qua 15 ảnh chụp

### 4.1 params0, det của lớp cũ và quyết định trên test S0 theo seed

Trong một FT seed, 5 seed dùng cùng ảnh S0 và cùng đặc trưng. Chỉ có hai thứ khác nhau: cơ sở PCA (05 §5.1) và việc chỉ số minibatch bốc ra ảnh nào (§2.1). Đo ba điều:
- số ảnh test S0 đoán đúng theo seed. Phép kiểm: phải trùng S0_All của 15 run (r0: 206, 206, 204, 205, 206; r1: 205, 205, 205, 203, 206; r2: 205, 208, 208, 207, 207);
- giữa hai seed bất kỳ, số ảnh test S0 bị đoán khác nhãn;
- det trung bình của 5 lớp cũ theo seed, tức `dets[0]` của luật dừng ở 07, cùng hai ngưỡng suy ra: lượt F dừng khi det lớp mới < det/2, lượt S dừng khi < det/0.707.

**Cách làm**: đọc `snap/04`, `snap/05`, `snap/06.pkl` của 15 run. Dự đoán bằng numpy như cell 4, trên `z_test0` của chính ảnh chụp đó. Hai seed của cùng r có cùng ảnh test S0; nếu thứ tự hàng của tập test khác nhau, đổi về cùng thứ tự bằng cách so nội dung từng hàng.

In [7]:
# cell 6: across the 15 snapshots (pickle): S0-test accuracy of params0 per seed (check: S0_All of the runs), pairwise label
#         disagreements on the S0 test set within each FT seed, mean det of the old classes and the F / S stop thresholds
import pickle
from itertools import combinations
snap_66 = {}
for d in sorted(Path("runs").glob("*__gc10_v2_ft_r[0-2]_s[0-4]")):
    tag = d.name.split("__")[1][len("gc10_v2_ft_"):]
    b4, b5, b6 = (pickle.loads((d / "snap" / f"{k}.pkl").read_bytes())["out"] for k in ("04", "05", "06"))
    p = b6["params0"]
    mu, S = np.asarray(p["class_means"], np.float64), np.asarray(p["class_covs"], np.float64)
    X = np.asarray(b5["z_test0"], np.float64)
    pred = np.stack([logpdf_44(X, mu[c], S[c]) for c in range(10)], 1).argmax(1)
    key = [r.tobytes() for r in np.asarray(b4["test0_x"])]             # raw test rows: image identity within an FT seed
    snap_66[tag] = dict(pred=dict(zip(key, pred)), y=dict(zip(key, np.asarray(b4["true_test0_y"]).astype(int))),
                        det=float(np.mean([np.linalg.det(S[c]) for c in range(5)])))
assert len(snap_66) == 15, len(snap_66)
for r in range(3):
    tags = [f"r{r}_s{s}" for s in range(5)]
    ok = [sum(snap_66[t]["pred"][k] == snap_66[t]["y"][k] for k in snap_66[t]["y"]) for t in tags]
    dis = [sum(snap_66[a]["pred"][k] != snap_66[b]["pred"][k] for k in snap_66[a]["pred"]) for a, b in combinations(tags, 2)]
    dets = [snap_66[t]["det"] for t in tags]
    print(f"r{r}: correct on S0 test {ok} | pairwise disagreements {min(dis)}-{max(dis)} of 241 | mean det old {[round(v, 4) for v in dets]}"
          f" | F threshold {round(min(dets) / 2, 3)}-{round(max(dets) / 2, 3)} | S threshold {round(min(dets) / 0.7071, 3)}-{round(max(dets) / 0.7071, 3)}")

r0: correct on S0 test [np.int64(206), np.int64(206), np.int64(204), np.int64(205), np.int64(206)] | pairwise disagreements 0-3 of 241 | mean det old [0.3708, 0.3704, 0.3723, 0.3719, 0.3707] | F threshold 0.185-0.186 | S threshold 0.524-0.526
r1: correct on S0 test [np.int64(205), np.int64(205), np.int64(205), np.int64(203), np.int64(206)] | pairwise disagreements 1-4 of 241 | mean det old [0.4192, 0.4174, 0.4206, 0.4188, 0.4194] | F threshold 0.209-0.21 | S threshold 0.59-0.595
r2: correct on S0 test [np.int64(205), np.int64(208), np.int64(208), np.int64(207), np.int64(207)] | pairwise disagreements 0-4 of 241 | mean det old [0.4254, 0.4235, 0.4264, 0.4246, 0.4251] | F threshold 0.212-0.213 | S threshold 0.599-0.603


- **Kết quả**:

  | FT seed | đúng trên test S0, seed 0–4 | số ảnh đoán khác nhau giữa 2 seed (trên 241) | det trung bình lớp cũ, seed 0–4 | ngưỡng F | ngưỡng S |
  |---|---|---|---|---|---|
  | r0 | 206, 206, 204, 205, 206 | 0–3 | 0.3708, 0.3704, 0.3723, 0.3719, 0.3707 | 0.185–0.186 | 0.524–0.526 |
  | r1 | 205, 205, 205, 203, 206 | 1–4 | 0.4192, 0.4174, 0.4206, 0.4188, 0.4194 | 0.209–0.210 | 0.590–0.595 |
  | r2 | 205, 208, 208, 207, 207 | 0–4 | 0.4254, 0.4235, 0.4264, 0.4246, 0.4251 | 0.212–0.213 | 0.599–0.603 |
- **Ý nghĩa**:
  - Phép kiểm đạt: số ảnh đúng trùng S0_All của cả 15 run.
  - Trong một FT seed, hai nguồn khác biệt (cơ sở PCA và minibatch) chỉ đổi quyết định ở tối đa 4/241 ảnh test S0. Chúng cũng chỉ đổi det trung bình của lớp cũ dưới 1% (r0: 0.3704–0.3723). Ở bước 06, seed phase 2 gần như không có tác dụng.
  - det của lớp cũ, và các ngưỡng dừng, khác nhau theo FT seed: r0 0.37, còn r1 và r2 khoảng 0.42–0.43. Ngưỡng S của r1, r2 (khoảng 0.59–0.60) cao hơn của r0 (0.52). [SUY LUẬN] Với cùng tốc độ giảm det của lớp mới, lượt S của r1 và r2 sẽ dừng sớm hơn r0. Khớp với bước dừng trong `metrics.json`: r0 526–620, r1 468–549, r2 475–507. Kiểm ở 07 và 13.
  - Vì §3.2 cho thấy det của lớp cũ phần lớn do ngân sách tối ưu quyết định, khác biệt giữa các FT seed (0.37 so với 0.42) là phần mà dữ liệu còn để lại trên nền đó. Chưa tách được dữ liệu đóng góp bao nhiêu.

## Summary (EN)

1. **Step 06 reproduces bit for bit on the GPU.** Rerunning the 1000 Adam steps of `gc10_v2_ft_r0_s4` in deterministic mode gives `params0` identical to the snapshot, despite a new GPU driver (§1.1). Slots 5–9 stay at mu = 0, Sigma = I; slots 0–4 move by at most 0.033 (mean) and 0.036 (covariance) per element (§1.1).
2. **"SVI" is maximum likelihood by Adam, limited by its step budget.** The guide is empty and every parameter is a `param`, so Trace_ELBO is the negative log-likelihood of a 128-image minibatch: no prior, no KL term (D11, §2.1). The learning rates sum to 0.0334 over 1000 steps; max |mu| reaches 0.0329, i.e. 98.5% of that budget (§2.2). The learned means are about 0.08 from the origin while the class centres are 8–11 away (r_mu 0.992–0.996), and Sigma keeps eigenvalues 0.82–1.32 while the sample covariances span 0.01–32 (D23, §3.1).
3. **The classifier is shaped in the warm-up, not by fitted Gaussians.** Train accuracy is 81.8% after one step and test accuracy settles at about 85% by step 100; the last 900 steps lower the loss but do not change the test score (§1.1). The "sign of the centre" classifier reaches 191/241 against 206/241 for params0, and the learned means align better with the centres than with their signs (cos 0.63–0.75 vs 0.59–0.68), so params0 is not that classifier (§3.1). [inference] Adam's per-coordinate normalisation may weight coordinates by a signal-to-noise ratio.
4. **det Sigma < 1 comes from the optimiser (Q06.05).** log det(I + E) ≈ tr E − ½‖E‖²_F holds to 0.03. The off-diagonal drift gives ½‖E_off‖² = 0.42–0.51 in every class (95% of ‖E‖²_F), and the trace adds −0.35 to −0.93. Together they give det 0.24–0.44 and a mean of 0.3707, the early-stop rule's `dets[0]` (§3.2). [inference] The stop thresholds are thus set mostly by the step budget (D14, D23).
5. **Seeds barely matter at step 06; FT seeds do.** Within an FT seed the 5 seeds disagree on at most 4/241 S0-test images and their mean old-class det varies by under 1%. Across FT seeds it is 0.37 (r0) against 0.42–0.43 (r1, r2), so the pass-S threshold is 0.52 against 0.59–0.60 (§4.1). [inference] This matches the earlier pass-S stops of r1 and r2 (468–549) versus r0 (526–620).